In [1]:
import os
from pathlib import Path
os.chdir(Path.cwd().parent)

In [2]:
from pathlib import Path
from tabulate import tabulate
import re

import numpy as np
import pandas as pd
import xarray as xr
import rioxarray as rxr

import rasterio
from rasterio.crs import CRS
from affine import Affine

from downscaling.settings_downscaling import CONVERSION_FACTORS

In [3]:
project_dir = Path.cwd()
print(f"Project directory: {project_dir}")

Project directory: k:\PythonWork\downscaling\Kaya_downscaling


# Check input population data

# ORIGINAL

In [4]:
# read in orginal data from dirs (tif files) and make xarray datasets
dir_pop_2UP_orginal = r"Z:\cold_data_storage\users\roelfsemam\surdrive_UU\NSA\Downscaling_share\input_data_downscaling\Population\processed\2UP\processed_GHSL_2024_M3\SSP2"
dir_pop_COMASS_orginal = r"Z:\cold_data_storage\users\roelfsemam\surdrive_UU\NSA\Downscaling_share\input_data_downscaling\Population\processed\COMPASS\processed_version_2\SSP2"
dirs_pop = [dir_pop_2UP_orginal, dir_pop_COMASS_orginal ]

dir_gdp_ppp_Murakami = r"Z:\cold_data_storage\users\roelfsemam\surdrive_UU\NSA\Downscaling_share\input_data_downscaling\GDP_PPP\processed\Murakami\processed_version_2021_1\SSP2"
dir_gdp_ppp_COMPASS = r"Z:\cold_data_storage\users\roelfsemam\surdrive_UU\NSA\Downscaling_share\input_data_downscaling\GDP_PPP\processed\COMPASS\processed_version_2\SSP2"
dirs_gdp = [dir_gdp_ppp_Murakami, dir_gdp_ppp_COMPASS]



In [5]:
search_pattern_pop_2UP = rf"M?var1_SSP2_(\d{{4}})_tpop\.tif"
search_pattern_pop_COMPASS = f"Population_count_(\\d{{4}})_6min_SSP2_not_harm.tif"
search_patterns = [search_pattern_pop_2UP, search_pattern_pop_COMPASS]
sources_pop = ["2UP", "COMPASS"]

search_pattern_gdp_ppp_Murakami = f"gdp(\\d{{4}}).tif"
search_pattern_gdp_ppp_COMPASS =  search_pattern = f"GDP_(\\d{{4}})_6min_SSP2_not_harm.tif"
search_patterns = [search_pattern_gdp_ppp_Murakami, search_pattern_gdp_ppp_COMPASS]
sources_gdp = ["Murakami", "COMPASS"]

In [6]:
driver = "GDP"

match driver:
    case "POP":
        dirs = dirs_pop
        search_pattern_base = search_pattern_pop_2UP
        search_pattern_sens = search_pattern_pop_COMPASS
        sources = sources_pop
        varname = "Population"
    case "GDP":
        dirs = dirs_gdp
        search_pattern_base = search_pattern_gdp_ppp_Murakami
        search_pattern_sens = search_pattern_gdp_ppp_COMPASS
        sources = sources_gdp
        varname = "GDP_PPP"

print(f"Processing {driver} data from sources: {sources}")
search_patterns = [search_pattern_base, search_pattern_sens]

df_SE_totals = pd.DataFrame()
for i, d in enumerate(dirs):
    files = list(Path(d).glob("*.tif"))
    data_list, years, opened = [], [], []
    for j, f in enumerate(files):
        print(f"Reading file: {f.name}")
        try:
            year = int(re.search(search_patterns[i], f.name).group(1))
        except AttributeError:
            print(f"Could not extract year from file: {f.name}")
            continue
        years.append(year)

        with rasterio.open(f, "r") as src:
            data = src.read(1)
            profile = src.profile.copy()

        if d == dir_pop_2UP_orginal:
            # Set/update profile parameters
            crs = CRS.from_epsg(4326)
            cell_size = (0.5/60)
            x_min = -180.0
            y_max = 90.0
            transform = Affine(cell_size, 0.0, x_min, 0.0, -cell_size, y_max)
            profile.update({
                "crs": crs,
                "transform": transform,
                "nodata": -9999
            })

        # Write output with updated profile
        output_tiff_dir = project_dir / "data" / "check" / "tif"
        output_tiff_dir.mkdir(parents=True, exist_ok=True)
        output_tiff_path = output_tiff_dir / f"processed_{f.name}"
        Path(output_tiff_path).unlink(missing_ok=True)
        with rasterio.open(output_tiff_path, "w", **profile) as dst:
            dst.write(data, 1)

        da = rxr.open_rasterio(output_tiff_path, masked=True, chunks={"x": "auto", "y": "auto"})
        opened.append(da)  # keep the raw handle so it can be closed in the finally block
        da = da.squeeze(drop=True)
        da = da.where(da >= 0)
        da = da.chunk("auto")
        data_list.append(da)
        da.close()

    print(f"Concatenating data for source: {sources[i]}")
    xr_SE = xr.concat(data_list, dim="time")
    xr_SE = xr_SE.assign_coords(time=years)
    xr_SE = xr_SE.to_dataset(name=varname)
    xr_SE = xr_SE.sortby("y", ascending=False)  # north-to-south
    xr_SE = xr_SE.sortby("x", ascending=True)  # west-to-east
    print(f"Summing data for source: {sources[i]}")
    xr_SE_total = xr_SE[varname].sum(dim=["y", "x"])
    df_SE_total = xr_SE_total.to_dataframe().reset_index()
    # convert to $2005 dollars
    if driver == "GDP" and sources[i] == "COMPASS":
        conv_factor = CONVERSION_FACTORS[(2017, 2005)]
        df_SE_total[varname] *= conv_factor
    df_SE_total["source"] = sources[i]
    df_SE_total = df_SE_total[df_SE_total["time"]>=2020]
    df_SE_totals = pd.concat([df_SE_totals, df_SE_total], ignore_index=True)

print(f"Pivoting data for sources: {sources}")
df_SE_totals = df_SE_totals.pivot(index="time", columns="source", values=varname).reset_index()
df_SE_totals.to_csv(project_dir / "data" / "check" / f"check_{varname.replace('|', '_').lower()}_original.csv", index=True, float_format="%.1f", sep=";")
#tabulate(df_SE_totals, headers="keys", tablefmt="grid", showindex=False, floatfmt=",.0f")
df_SE_totals


Processing GDP data from sources: ['Murakami', 'COMPASS']
Reading file: gdp1850.tif
Reading file: gdp1860.tif
Reading file: gdp1870.tif
Reading file: gdp1880.tif
Reading file: gdp1890.tif
Reading file: gdp1900.tif
Reading file: gdp1910.tif
Reading file: gdp1920.tif
Reading file: gdp1930.tif
Reading file: gdp1940.tif
Reading file: gdp1950.tif
Reading file: gdp1960.tif
Reading file: gdp1970.tif
Reading file: gdp1980.tif
Reading file: gdp1990.tif
Reading file: gdp2000.tif
Reading file: gdp2010.tif
Reading file: gdp2020.tif
Reading file: gdp2030.tif
Reading file: gdp2040.tif
Reading file: gdp2050.tif
Reading file: gdp2060.tif
Reading file: gdp2070.tif
Reading file: gdp2080.tif
Reading file: gdp2090.tif
Reading file: gdp2100.tif
Concatenating data for source: Murakami
Summing data for source: Murakami
Reading file: GDP_2015_6min_SSP2_not_harm.tif
Reading file: GDP_2020_6min_SSP2_not_harm.tif
Reading file: GDP_2025_6min_SSP2_not_harm.tif
Reading file: GDP_2030_6min_SSP2_not_harm.tif
Reading 

source,time,COMPASS,Murakami
0,2020,1.048438e+14,1.012073e+14
1,2025,1.258212e+14,NaN
2,2030,1.466388e+14,1.431129e+14
3,2035,1.683383e+14,NaN
4,2040,1.901990e+14,1.860473e+14
5,2045,2.131553e+14,NaN
6,2050,2.372654e+14,2.313644e+14
7,2055,2.626775e+14,NaN
8,2060,2.899138e+14,2.805525e+14
9,2065,3.189490e+14,NaN


# Downscaled

In [7]:
profile = "base_run"

In [8]:
xr_pop_downscaled = xr.open_dataset(project_dir / "data" / "processed" / profile / f"Population_downscaled_2UP_GHSL_2024_M3_SSP2_cf_12.nc")
print(f"CRS: {xr_pop_downscaled.rio.crs}")
xr_pop_downscaled

FileNotFoundError: [Errno 2] No such file or directory: 'k:\\PythonWork\\downscaling\\Kaya_downscaling\\data\\processed\\base_run\\Population_downscaled_2UP_GHSL_2024_M3_SSP2_cf_12.nc'

In [ ]:
xr_gdp_ppp_downscaled = xr.open_dataset(project_dir / "data" / "processed" / profile / f"GDP_PPP_downscaled_Murakami_version_2021_1_SSP2_cf_1.2.nc")
print(f"CRS: {xr_gdp_ppp_downscaled.rio.crs}")
xr_gdp_ppp_downscaled

## PROCESSED

In [ ]:
dir_2UP_processed = "K:/PythonWork/downscaling/Kaya_downscaling/data/processed/base_run"
dir_COMPAS_processed = "K:/PythonWork/downscaling/Kaya_downscaling/data/processed/sensitivity_4"
file_2UP_processed = "Population_2UP_GHSL_2024_M3_SSP2_cf_12.nc"
file_COMPASS_processed = "Population_COMPASS_version_2_SSP2_cf_1.nc"

In [ ]:
xr_pop_2UP_processed = xr.open_dataset(Path(dir_2UP_processed) / file_2UP_processed)
xr_pop_COMPASS_processed = xr.open_dataset(Path(dir_COMPAS_processed) / file_COMPASS_processed)

In [ ]:
xr_pop_2UP_processed_total = xr_pop_2UP_processed["Population"].sum(dim=["y", "x"])
df_SE_2UP_processed_total = xr_pop_2UP_processed_total.to_dataframe().reset_index()
df_SE_2UP_processed_total["source"] = "2UP"
df_SE_2UP_processed_total.rename(columns={"time": "year"}, inplace=True)
df_SE_2UP_processed_total.drop(columns=["spatial_ref"], inplace=True, errors="ignore")
xr_pop_COMPASS_processed_total = xr_pop_COMPASS_processed["Population"].sum(dim=["y", "x"])
df_SE_COMPASS_processed_total = xr_pop_COMPASS_processed_total.to_dataframe().reset_index()
df_SE_COMPASS_processed_total["source"] = "COMPASS"
df_SE_COMPASS_processed_total.rename(columns={"time": "year"}, inplace=True)
df_SE_COMPASS_processed_total.drop(columns=["spatial_ref"], inplace=True, errors="ignore")

In [ ]:
df_SE_processed = pd.concat([df_SE_2UP_processed_total, df_SE_COMPASS_processed_total], ignore_index=True)
df_SE_processed = df_SE_processed.pivot(index="year", columns="source", values="Population").reset_index()
df_SE_processed.to_csv(project_dir / "data" / "check" / f"check_population.csv", index=True, float_format="%.1f", sep=";")
df_SE_processed